# BMD-45 Finale (Drive-native — joint polish + full-val report)

Closes the 8-loop chain (0.7949 → 0.8293). One session, **no HF download**:
1. Mount Drive → copy `bmd_contract_live` → `/tmp/bmd_finale` (~10 min).
2. Polish: 5 epochs from loop-8 `best.pt` at low LR over the full pool.
3. Report on the FULL official val (~10k images), not the 3.4k anchor.
4. Package `india-yolov8n-final` + output pack zip for the workspace.

Expected: +0.002–0.005 overall and a smoother tail — polish, not
breakthrough. If overall drops >0.01 vs 0.8293, the polish hurt: keep
the loop-8 registry as the winner and say so in the handoff.

## Runtime
T4 GPU, one session (~3-4h total). Tab focused, cells staged.

## 0. Params — edit once

In [ ]:
# ---- edit ----------------------------------------------------------
PREV_WEIGHTS = "/content/best_f007.pt"  # loop-8 best.pt, uploaded below
EPOCHS = 5
LR0 = 0.002  # low-LR polish; do not raise without evidence
# ---- stable --------------------------------------------------------
HF_REPO = "iisc-aim/BMD-45"
HF_TRAIN = "BMD-45-Train"
HF_VAL = "BMD-45-Val"
TRAIN_FOLDERS = ["images_%03d" % i for i in range(8)]
VAL_FOLDERS = ["images_%03d" % i for i in range(3)]
DRIVE_CONTRACT = "/content/drive/MyDrive/bmd_contract_live"  # JPG+labels
DRIVE_RAW = "/content/drive/MyDrive/bmd45_raw"  # optional PNG mirror
ROOT = "/tmp/bmd_finale"

print('init:', PREV_WEIGHTS, '| epochs:', EPOCHS, '| lr0:', LR0)

## 1. Setup

Uploads needed (file browser): this notebook + `best_f007.pt` (~6 MB).
Pinned deps, Drive mount, HF auth (for metadata only), ~30 GB disk guard.

In [ ]:
!pip install -q ultralytics onnx onnxruntime huggingface_hub

import os
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
assert os.path.isfile(PREV_WEIGHTS), 'upload best_f007.pt first: ' + PREV_WEIGHTS

try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        print('HF auth: token loaded')
    else:
        print('HF auth: anonymous')
except Exception as e:
    print('HF auth: userdata unavailable (%s), anonymous' % e)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_CONTRACT = '/content/drive/MyDrive/bmd_contract_live'
DRIVE_RAW = '/content/drive/MyDrive/bmd45_raw'
assert os.path.isdir(DRIVE_CONTRACT), 'Drive contract missing: ' + DRIVE_CONTRACT

import shutil
free_gb = shutil.disk_usage('/tmp').free / 1e9
print('/tmp free: %.1f GB' % free_gb)
assert free_gb > 30, 'need ~30 GB free; free space and retry'
os.makedirs(ROOT, exist_ok=True)


## 2. Drive → /tmp (fast intra-DC copy, ~10 min)

Copies `bmd_contract_live` → `/tmp/bmd_finale`. Skips existing files. Rewrites `data.yaml` with absolute paths. Validates counts.

In [ ]:
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
import os, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

SRC = Path(DRIVE_CONTRACT)
DST = Path(ROOT)
assert SRC.is_dir(), 'Drive contract missing: ' + str(SRC)

jobs=[]
for root, _, files in os.walk(SRC):
    for f in files:
        s = Path(root) / f
        d = DST / s.relative_to(SRC)
        try:
            if d.exists() and d.stat().st_size == s.stat().st_size:
                continue
        except OSError:
            pass
        jobs.append((s,d))

print(f'copying {len(jobs)} files Drive -> /tmp (skipping {len(list(SRC.rglob("*")))-len(jobs)} existing)...')
if jobs:
    def _one(pair):
        s,d = pair
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(s, d)
        return 1
    done=0
    with ThreadPoolExecutor(max_workers=32) as ex:
        for _ in ex.map(_one, jobs):
            done += 1
            if done % 5000 == 0:
                print(f'  {done}/{len(jobs)} ...')
    print(f'done: {done} copied')
else:
    print('nothing to copy (already synced)')

yaml_src = SRC / 'data.yaml'
yaml_dst = Path(ROOT) / 'data.yaml'
if yaml_src.is_file():
    txt = yaml_src.read_text(encoding='utf-8')
    lines = txt.split('\n')
    out = []
    for ln in lines:
        if ln.strip().startswith('path:'):
            out.append('path: ' + str(Path(ROOT).resolve()))
        else:
            out.append(ln)
    yaml_dst.write_text('\n'.join(out) + '\n', encoding='utf-8')
    print('data.yaml rewritten with absolute paths')
else:
    CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto']
    lines = ['path: ' + str(Path(ROOT).resolve()), 'train: images/train',
             'val: images/val', 'test: images/val', 'names:']
    lines += ['  %d: %s' % (i, n) for i, n in enumerate(CLASSES)]
    Path(ROOT, 'data.yaml').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print('data.yaml created')

ntr = len(list(Path(ROOT, 'images', 'train').glob('*.jpg')))
nva = len(list(Path(ROOT, 'images', 'val').glob('*.jpg')))
print('contract: train %d images, val %d images' % (ntr, nva))
assert ntr > 30000 and nva > 9000, 'incomplete contract — still syncing?'


## 3. Polish (5 epochs, low LR, full pool)

In [ ]:
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
import os
from ultralytics import YOLO

pool = list(__import__('pathlib').Path(ROOT, 'images', 'train').glob('*.jpg'))
print('train pool: %d images' % len(pool))
assert len(pool) > 30000, 'train pool incomplete — check Drive sync'
print('init weights:', PREV_WEIGHTS)
model = YOLO(PREV_WEIGHTS)
model.train(
    data=os.path.join(ROOT, 'data.yaml'),
    epochs=EPOCHS, imgsz=640, batch=32, patience=8, workers=4,
    lr0=LR0, name='finale',
)  # GPU OOM? batch=16. Host-RAM kill? workers=2.

run_dir = model.trainer.save_dir
BEST = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', BEST)


## 4. Full-val report + per-class table (the publication-grade number)

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Polish) first'
import os
from ultralytics import YOLO

metrics = YOLO(BEST).val(data=os.path.join(ROOT, 'data.yaml'), verbose=False)
names = metrics.names
print('%-10s %9s %9s %9s' % ('class', 'precision', 'recall', 'mAP50'))
per_class = {}
for i, ci in enumerate(metrics.box.ap_class_index):
    nm = names[int(ci)]
    m = round(float(metrics.box.ap50[i]), 4)
    per_class[nm] = m
    print('%-10s %9.3f %9.3f %9.3f' % (nm, float(metrics.box.p[i]), float(metrics.box.r[i]), m))
mAP50 = round(float(metrics.box.map50), 4)
print('FINALE overall mAP50 (full val): %.4f' % mAP50)
print('loop-8 anchor baseline was 0.8293 — delta: %+.4f' % (mAP50 - 0.8293))


## 5. Export + static int8 + final registry

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Polish) first'
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto']
import glob, os
import shutil as _sh, traceback
import cv2 as _cv2, numpy as _np
from onnxruntime.quantization import CalibrationDataReader as _CDR, quantize_static as _qs, QuantType as _QT

onnx_path = YOLO(BEST).export(format='onnx', imgsz=640, opset=17, simplify=True)
print('onnx:', onnx_path)

calib_imgs = sorted(__import__('glob').glob(os.path.join(ROOT, 'images', 'val', '*.jpg')))[:200]
assert calib_imgs, 'no val images for calibration'

class _FR(_CDR):
    def __init__(self, frames, onx):
        import onnxruntime as _ort
        self.input_name = _ort.InferenceSession(onx, providers=['CPUExecutionProvider']).get_inputs()[0].name
        self.reiter = iter([self._pre(p) for p in frames])
    def _pre(self, path):
        img = _cv2.imread(path)
        img = _cv2.resize(img, (640, 640))
        rgb = _cv2.cvtColor(img, _cv2.COLOR_BGR2RGB).astype(_np.float32) / 255.0
        return {self.input_name: _np.transpose(rgb, (2, 0, 1))[_np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

int8_path = onnx_path.replace('.onnx', '-int8.onnx')
try:
    _qs(onnx_path, int8_path, _FR(calib_imgs, onnx_path), weight_type=_QT.QInt8)
    print('STATIC int8 ->', int8_path)
except Exception:
    traceback.print_exc()
    raise

import datetime, json
reg = os.path.join(ROOT, 'registry', 'india-yolov8n-final')
os.makedirs(reg, exist_ok=True)
_sh.copy(onnx_path, os.path.join(reg, 'model.onnx'))
_sh.copy(int8_path, os.path.join(reg, 'model-int8.onnx'))
meta = {'name': 'india-yolov8n-final', 'classes': CLASSES, 'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255], 'layout': 'NCHW', 'color': 'RGB'},
        'quantization': 'int8', 'source_run': datetime.date.today().isoformat(),
        'metrics': {'mAP50_overall': mAP50, 'per_class_mAP50': per_class},
        'provenance': {'hf_repo': HF_REPO, 'run': 'finale-joint-polish',
                       'init_weights': 'loop-8 best.pt', 'epochs': EPOCHS, 'lr0': LR0,
                       'mapping': 'optionB-merge-6class', 'val': 'official-full-10k'}}
open(os.path.join(reg, 'metadata.json'), 'w').write(json.dumps(meta, indent=2))
print('registry:', reg)


## 6. Finale pack (auto-download — drop this zip in the workspace)

In [ ]:
assert 'BEST' in dir() and 'mAP50' in dir(), 'run cells 3-4 first'
import datetime, os, shutil

pack_name = 'bmd_finale_%s' % datetime.date.today().isoformat()
pack_dir = os.path.join(ROOT, pack_name)
os.makedirs(pack_dir, exist_ok=True)
shutil.copytree(os.path.join(ROOT, 'registry', 'india-yolov8n-final'),
                os.path.join(pack_dir, 'registry'), dirs_exist_ok=True)
shutil.copy2(BEST, os.path.join(pack_dir, 'best.pt'))
results_csv = os.path.join(os.path.dirname(BEST), '..', 'results.csv')
if os.path.isfile(results_csv):
    shutil.copy2(results_csv, pack_dir)
fh = open(os.path.join(pack_dir, 'finale_report.txt'), 'w')
fh.write('finale mAP50 (full val): %.4f\n' % mAP50)
fh.write('per-class: %s\n' % {k: round(v, 4) for k, v in per_class.items()})
fh.write('delta vs loop-8 anchor (0.8293): %+.4f\n' % (mAP50 - 0.8293))
fh.close()
zip_path = shutil.make_archive(os.path.join(ROOT, pack_name), 'zip', root_dir=ROOT, base_dir=pack_name)
print('pack: %s (%.1f MB)' % (zip_path, os.path.getsize(zip_path) / 1e6))
from google.colab import files
files.download(zip_path)
print('DONE — place %s.zip in notebooks/training_output_zips/.' % pack_name)
print('If finale mAP50 < 0.8193: keep loop-8 registry as winner, report it as such.')